# Data Cleaning and Validation
### Dataset: Restaurant data (`Dataset.csv`)

In this notebook I go through the dataset step by step to clean it up and check that the values make sense.

**Steps:**
1. Load the data
2. Check for missing values
3. Check for duplicate rows
4. Fix a currency text encoding issue
5. Clean up extra spaces in text columns
6. Validate that number columns are in a sensible range
7. Save a clean copy of the dataset


## Step 1: Load the dataset

In [1]:
# Import Libraries
import pandas as pd
import numpy as np

# Display Setting
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [2]:
# Read the CSV File
df = pd.read_csv('Dataset.csv')
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,Average Cost for two,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenue, Poblacion, Makati City","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Makati City",121.027535,14.565443,"French, Japanese, Desserts",1100,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Makati City",121.014101,14.553708,Japanese,1200,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Mandaluyong City",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",4000,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandaluyong City",121.056475,14.585318,"Japanese, Sushi",1500,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandaluyong City",121.057508,14.584450,"Japanese, Korean",1500,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


## Step 2. Inspecting the Dataset

In [3]:
# Shape of the Dataset
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

Rows: 9551, Columns: 21


In [4]:
# Data types of Columns
df.dtypes

Restaurant ID             int64
Restaurant Name             str
Country Code              int64
City                        str
Address                     str
Locality                    str
Locality Verbose            str
Longitude               float64
Latitude                float64
Cuisines                    str
Average Cost for two      int64
Currency                    str
Has Table booking           str
Has Online delivery         str
Is delivering now           str
Switch to order menu        str
Price range               int64
Aggregate rating        float64
Rating color                str
Rating text                 str
Votes                     int64
dtype: object

In [5]:
# Statistical Summary
df.describe()

,Restaurant ID,Country Code,Longitude,Latitude,Average Cost for two,Price range,Aggregate rating,Votes
count,9.551000e+03,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000
mean,9.051128e+06,18.365616,64.126574,25.854381,1199.210763,1.804837,2.666370,156.909748
std,8.791521e+06,56.750546,41.467058,11.007935,16121.183073,0.905609,1.516378,430.169145
min,5.300000e+01,1.000000,-157.948486,-41.330428,0.000000,1.000000,0.000000,0.000000
25%,3.019625e+05,1.000000,77.081343,28.478713,250.000000,1.000000,2.500000,5.000000
50%,6.004089e+06,1.000000,77.191964,28.570469,400.000000,2.000000,3.200000,31.000000
75%,1.835229e+07,1.000000,77.282006,28.642758,700.000000,2.000000,3.700000,131.000000
max,1.850065e+07,216.000000,174.832089,55.976980,800000.000000,4.000000,4.900000,10934.000000


## Step 3. Check for Missing Values

In [6]:
# Check columns with missing values
print(df.isnull().sum()[df.isnull().sum() > 0])

Cuisines    9
dtype: int64


> The `Cuisines` column has a few missing values. Instead of deleting these rows, we fill them with **Not Specified** so we don't lose real restaurant records.

In [7]:
df['Cuisines'] = df['Cuisines'].fillna('Not Specified')

# Check again to confirm it worked
df['Cuisines'].isnull().sum()

np.int64(0)

## Step 3: Check and remove duplicate rows

In [8]:
duplicate_count = df.duplicated().sum()
print('Number of Duplicate Records found: ', duplicate_count)

# Also check duplicates based on Restaurent ID (should be unique)
duplicate_ids = df['Restaurant ID'].duplicated().sum()
print('Duplicate Restaurant ID found: ', duplicate_ids)

Number of Duplicate Records found:  0
Duplicate Restaurant ID found:  0


## Step 4: Fix broken text in the Currency column & City column

> **When the file was saved, the British Pound symbol (£) got corrupted and shows up as strange characters. We fix that here.**

In [9]:
df['Currency'].unique()

<StringArray>
[      'Botswana Pula(P)',     'Brazilian Real(R$)',              'Dollar($)',
     'Emirati Diram(AED)',     'Indian Rupees(Rs.)', 'Indonesian Rupiah(IDR)',
          'NewZealand($)',             'Pounds(��)',        'Qatari Rial(QR)',
                'Rand(R)',  'Sri Lankan Rupee(LKR)',       'Turkish Lira(TL)']
Length: 12, dtype: str

In [10]:
# Replace corrupted pound symbol with the correct £ symbol
df['Currency'] = df['Currency'].str.replace('Pounds(��)', 'Pounds(£)', regex=False)

# Confirm the fix
df[df['Currency'].str.contains('Pounds')]['Currency'].unique()

<StringArray>
['Pounds(£)']
Length: 1, dtype: str

> **When the file was saved, some city names containing special/non-ASCII characters were corrupted due to an encoding issue, causing characters such as `İ` in `İstanbul` to appear as `��`. We fix that here.**

In [11]:
df[df['City'].str.contains('�', na=False)]['City'].unique()

<StringArray>
['Bras�_lia', 'S��o Paulo', '��stanbul']
Length: 3, dtype: str

In [12]:
city_corrections = {
    'Bras�_lia': 'Brasília',
    'S��o Paulo': 'São Paulo',
    '��stanbul': 'İstanbul'
}

df['City'] = df['City'].replace(city_corrections)

In [13]:
# Verify corrected city names
df[df['City'].isin(city_corrections.values())]['City'].unique()

<StringArray>
['Brasília', 'São Paulo', 'İstanbul']
Length: 3, dtype: str

## Step 5: Standardize Column Names

In [14]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

## Step 6: Clean up text columns (remove extra spaces)

In [15]:
text_columns = ['restaurant_name', 'city', 'address', 'locality', 'cuisines']

for col in text_columns:
    df[col] = df[col].str.strip()

print('Text Columns Cleaned!')

Text Columns Cleaned!


## Step 7: Validate number columns are within a sensible range

In [16]:
# Aggregate rating should always be between 0 and 5
invalid_ratings = df[(df['aggregate_rating'] < 0) | (df['aggregate_rating'] > 5)]
print('Rows with an invalid rating (outside 0-5):', len(invalid_ratings))

# Price range should only be 1, 2, 3, or 4
invalid_price_range = df[~df['price_range'].isin([1, 2, 3, 4])]
print('Rows with an invalid price range:', len(invalid_price_range))

# Votes and Average Cost for two should not be negative
invalid_votes = df[df['votes'] < 0]
invalid_cost = df[df['average_cost_for_two'] < 0]
print('Rows with negative votes:', len(invalid_votes))
print('Rows with negative average cost:', len(invalid_cost))

Rows with an invalid rating (outside 0-5): 0
Rows with an invalid price range: 0
Rows with negative votes: 0
Rows with negative average cost: 0


(In this dataset none of these problems were found, but the checks are kept here so the data gets validated every time this notebook runs.)

## Step 8: Save the cleaned dataset

In [17]:
df.to_csv('Dataset_cleaned.csv', index=False)

print('Cleaned file saved as Dataset_cleaned.csv')

Cleaned file saved as Dataset_cleaned.csv


## Summary

- Filled 9 missing `Cuisines` values with `"Not Specified"`
- Confirmed there were no duplicate rows or duplicate Restaurant IDs
- Fixed a corrupted £ symbol in the `Currency` column
- Fixed corrupted city names caused by encoding issues, including restoring special characters such as İ in İstanbul.
- Removed extra spaces from text columns
- Validated that ratings, price range, votes, and cost values were all within sensible ranges
- Saved the cleaned data to `Dataset_cleaned.csv`
